# Chapter 10: Natural Language Processing Basics
## Notebook 03 — Advanced NLP

This notebook introduces **attention mechanisms**, **sequence-to-sequence** concepts, **transfer learning** in NLP (e.g. BERT), **multi-task systems**, **language generation** basics, and **production considerations**. It sets the stage for **Chapter 11: Large Language Models & Transformers**.

### What you'll learn

| Topic | Section |
|-------|--------|
| Attention in NLP | §1 |
| Seq2seq and applications | §2 |
| Transfer learning (BERT/DistilBERT) | §3 |
| Multi-task NLP system | §4 |
| Language generation basics | §5 |
| Production considerations | §6 |
| Preview of Chapter 11 & capstone | §7–8 |

**Estimated time:** 2.5–3 hours

---
*Generated by Berta AI*

---
## 1. Attention Mechanisms in NLP

**Attention** lets the model focus on different parts of the input when producing each output. **Self-attention** (as in Transformers) computes relationships between all positions. We implement a simple **scaled dot-product attention** from scratch.

In [ ]:
import numpy as np

def scaled_dot_product_attention(Q, K, V):
    """Q, K, V: (seq_len, d_k). Returns (seq_len, d_v) and attention weights."""
    d_k = Q.shape[1]
    scores = np.dot(Q, K.T) / np.sqrt(d_k)
    weights = np.exp(scores - scores.max(axis=1, keepdims=True))
    weights = weights / weights.sum(axis=1, keepdims=True)
    out = np.dot(weights, V)
    return out, weights

np.random.seed(42)
seq_len, d = 4, 8
Q = np.random.randn(seq_len, d).astype(np.float32)
K = np.random.randn(seq_len, d).astype(np.float32)
V = np.random.randn(seq_len, d).astype(np.float32)
out, attn = scaled_dot_product_attention(Q, K, V)
print("Output shape:", out.shape)
print("Attention weights (row sums = 1):")
print(np.round(attn, 3))

---
## 2. Sequence-to-Sequence Models

**Encoder-decoder**: encoder reads the source sequence into a context vector; decoder generates the target step-by-step. **Attention** in seq2seq lets the decoder look back at encoder states (e.g. for machine translation). Full implementation is in Chapter 11 with Transformers.

---
## 3. Transfer Learning in NLP

**Pre-trained language models** (BERT, DistilBERT, RoBERTa) are fine-tuned on your task (classification, NER, QA). Below we show a **conceptual** call using `transformers` if installed; otherwise we skip and describe the workflow.

In [ ]:
try:
    from transformers import pipeline
    classifier = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")
    result = classifier("This movie is great!")
    print("DistilBERT sentiment:", result)
except Exception as e:
    print("Transformers not used (install with: pip install transformers).")
    print("Fine-tuning workflow: load pre-trained model -> add task head -> train on your labels -> evaluate.")

---
## 4. Advanced Project: Multi-task NLP System

Design a system that does **text classification**, **NER**, and **sentiment** from the same text. Share an embedding layer across tasks (e.g. one encoder, multiple heads). Evaluation: per-task metrics (accuracy/F1 for classification, entity-level F1 for NER).

---
## 5. Language Generation Basics

**Autoregressive** generation: predict next token given previous tokens. **Sampling**: sample from the model's distribution (with **temperature** to control randomness). **Top-k**: sample from top k tokens. **Greedy**: always pick argmax. Chapter 11 covers this in depth with LLMs.

---
## 6. Production Considerations

- **Serialization**: Save vectorizer + model (e.g. pickle, joblib, or framework-specific).
- **Inference**: Batch inputs, fixed max length, GPU if available.
- **Error handling**: Validate input length/encoding; return graceful errors.
- **Monitoring**: Log latency, throughput, and error rates.

Example: save/load with joblib for a sklearn pipeline.

In [ ]:
import os
import joblib
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

pipe = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000)),
    ('clf', LogisticRegression(max_iter=500))
])
pipe.fit(["good", "bad"], [1, 0])
path = "../models/sentiment_pipeline.joblib"
os.makedirs(os.path.dirname(path), exist_ok=True)
joblib.dump(pipe, path)
loaded = joblib.load(path)
print("Loaded pipeline predict:", loaded.predict(["great"]))

---
## 7. Advanced Topics Preview

- **Chapter 11 — Transformers**: Self-attention, multi-head attention, encoder-decoder, BERT/GPT architecture.
- **Chapter 11 — LLMs**: Scaling, prompting, and applications.
- **Chapter 12 — Prompt engineering**: Designing prompts for LLMs.
- **Chapter 14 — Fine-tuning at scale**: LoRA, full fine-tuning, data and compute.

---
## 8. Capstone Project Design

Build an **end-to-end NLP application**: e.g. chatbot, summarizer, question answerer, or sentiment analyzer. Steps: **data prep** → **model building** (or fine-tuning) → **evaluation** → **deployment prep** (API, Docker). Choose one and walk through; then build your own variant.

---
*Generated by Berta AI*